# Generate ERP Airlines Source Data

## Purpose

This notebook validates the AeroPulse synthetic ERP airline data generator.

The generator simulates an upstream ERP source system and creates airline master-data records.

### Source Details

- Source system: ERP
- Entity: Airlines
- Format: CSV
- Environment: Development

### Target Landing Location

`/Volumes/workspace/aeropulse_dev/raw_landing/erp/airlines/`

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/source_generators/airline_generator.py

In [0]:
airlines_df = generate_airlines(
    spark=spark,
    record_count=100,
)

In [0]:
display(airlines_df)

In [0]:
print(f"Generated record count: {airlines_df.count()}")

In [0]:
print(f"Generated record count: {airlines_df.count()}")

# Land Airlines Source Data in the Raw Landing Volume

The generated airline source data will be written as CSV files to the AeroPulse development raw landing volume.

Target path:

`/Volumes/workspace/aeropulse_dev/raw_landing/erp/airlines/`

In [0]:
ENVIRONMENT = "dev"

CATALOG = "workspace"

SCHEMA = f"aeropulse_{ENVIRONMENT}"

RAW_LANDING_VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing"
)

AIRLINES_LANDING_PATH = (
    f"{RAW_LANDING_VOLUME_PATH}/erp/airlines"
)

print(AIRLINES_LANDING_PATH)

In [0]:
(
    airlines_df.write
    .mode("overwrite")
    .option("header", "true")
    .csv(AIRLINES_LANDING_PATH)
)

print("Airlines source data landed successfully.")

In [0]:
display(
    dbutils.fs.ls(AIRLINES_LANDING_PATH)
)

In [0]:
landed_airlines_df = (
    spark.read
    .option("header", "true")
    .csv(AIRLINES_LANDING_PATH)
)

display(landed_airlines_df)

In [0]:
print(
    f"Landed record count: {landed_airlines_df.count()}"
)

In [0]:
source_count = airlines_df.count()

landed_count = landed_airlines_df.count()

print(f"Source count: {source_count}")
print(f"Landed count: {landed_count}")

assert source_count == landed_count, (
    "Source and landed record counts do not match."
)

print("Source and landed record counts match successfully.")

# Create a Timestamped ERP Source Delivery

This section creates a uniquely identified source delivery in the AeroPulse raw landing layer.

Each delivery represents a separate batch received from the simulated ERP source system.

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/source_landing/source_file_landing.py

In [0]:
ENVIRONMENT = "dev"

CATALOG = "workspace"

SCHEMA = f"aeropulse_{ENVIRONMENT}"

SOURCE_SYSTEM = "erp"

SOURCE_ENTITY = "airlines"

RAW_LANDING_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing"
)

SOURCE_LANDING_DIRECTORY = (
    f"{RAW_LANDING_PATH}/"
    f"{SOURCE_SYSTEM}/"
    f"{SOURCE_ENTITY}"
)

print(SOURCE_LANDING_DIRECTORY)

In [0]:
from datetime import datetime, timezone

airlines_df = generate_airlines(
    spark=spark,
    record_count=100,
)

# Create timestamped delivery path
batch_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
delivery_filename = f"{SOURCE_ENTITY}_{batch_timestamp}.csv"
delivery_path = f"{SOURCE_LANDING_DIRECTORY}/{delivery_filename}"

# Write to timestamped CSV file using 'error' mode (Spark Connect compatible)
(
    airlines_df.write
    .mode("error")
    .option("header", "true")
    .csv(delivery_path)
)

print(
    f"Source delivery created successfully:\n"
    f"{delivery_path}"
)

In [0]:
display(
    dbutils.fs.ls(SOURCE_LANDING_DIRECTORY)
)

In [0]:
display(
    dbutils.fs.ls(delivery_path)
)

In [0]:
delivery_df = (
    spark.read
    .option("header", "true")
    .csv(delivery_path)
)

display(delivery_df)

In [0]:
delivery_count = delivery_df.count()

print(
    f"Delivery record count: {delivery_count}"
)

assert delivery_count == 100

print(
    "Source delivery validated successfully."
)